In [17]:
# Importamos librerias
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from IPython.display import display

In [18]:
# Importamos el dataset
df = pd.read_csv("data/Gaming_Academic_performance/Gaming_Academic_Performance.csv", index_col="student_id")
display(df)

,age,gender,gaming_hours,study_hours,sleep_hours,attendance,gaming_genre,social_activity,device_usage,reaction_time_ms,addiction_score,stress_level,grades
student_id,,,,,,,,,,,,,
1,22,Male,7.23,8.78,6.96,91.44,FPS,3.25,9.36,235.84,14.69,Low,86.459555
2,19,Male,0.07,8.72,7.63,63.63,Casual,1.02,3.21,328.71,2.47,Medium,98.230000
3,23,Female,1.73,9.56,4.40,83.26,Casual,3.46,5.56,313.61,4.73,High,90.560000
4,20,Female,6.62,1.68,7.83,75.04,RPG,1.46,11.78,241.84,14.54,Low,32.670000
5,22,Female,5.36,5.83,5.55,65.57,FPS,1.01,8.23,249.31,12.48,Low,58.710000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7991,20,Male,6.02,5.29,6.73,64.56,Casual,4.77,7.43,231.47,11.63,Low,39.590000
7992,22,Female,1.22,9.14,4.34,93.97,Casual,4.81,4.22,308.25,5.41,High,105.211520
7993,21,Female,3.17,6.18,6.54,69.78,Casual,0.95,7.22,288.80,7.07,Medium,81.160000


In [19]:
# Separamos varibables categoricas de numericas
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Codificamos variables categoricas como enteros (0 a n)
le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

print(numeric_cols)
print(categorical_cols)
display(df[categorical_cols].head())

['age', 'gaming_hours', 'study_hours', 'sleep_hours', 'attendance', 'social_activity', 'device_usage', 'reaction_time_ms', 'addiction_score', 'grades']
['gender', 'gaming_genre', 'stress_level']


/var/folders/8q/kpr1c3bd783bcqg85qsy39hr0000gn/T/ipykernel_30735/2265321866.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()


,gender,gaming_genre,stress_level
student_id,,,
1,1,1,1
2,1,0,2
3,0,0,0
4,0,2,1
5,0,1,1


In [20]:
# limpiamos el dataset
# Separamos target antes de escalar
cols_to_scale = [c for c in numeric_cols if c != "grades"]
X = df[cols_to_scale + categorical_cols]
y = df["grades"]

# Separamos train/test y val
# Validacion (ultimos 5 datos)
X_val = X.tail(5)
y_val = y.tail(5)

# Eliminamos los ultimos 5 datos (validacion) del dataset
X = X.iloc[:-5]
y = y.iloc[:-5]

# Test/train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Escalado de variables (solo numericas, categoricas no se escalan)
scaler = StandardScaler()
X_train = X_train.copy()
X_test  = X_test.copy()
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale]  = scaler.transform(X_test[cols_to_scale])
X_val[cols_to_scale]  = scaler.transform(X_val[cols_to_scale])

# Mostramos el dataset limpio
display(X_train)

,age,gaming_hours,study_hours,sleep_hours,attendance,social_activity,device_usage,reaction_time_ms,addiction_score,gender,gaming_genre,stress_level
student_id,,,,,,,,,,,,
3568,1.553750,0.995639,-0.394071,-0.231991,-0.286381,-1.417568,0.057784,-0.465917,1.436228,1,0,1
2355,0.781736,-0.313683,-1.396029,-0.810024,-0.350436,0.994871,-0.346481,0.167793,-0.312221,0,2,2
2652,1.167743,0.290954,-0.176592,0.701219,-0.540870,-1.014336,0.610402,-0.191207,0.257969,0,2,2
4129,0.781736,0.599798,-1.726132,0.415685,-0.823925,0.695924,1.300247,-0.612656,0.740437,1,2,1
1451,-1.534305,0.817293,-0.646502,1.571751,-0.697546,0.772399,1.500524,-0.731413,0.927842,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5227,-0.376284,0.138708,-0.658153,-0.823952,1.040599,1.113060,-0.253759,-0.160835,0.299836,0,0,2
5391,1.553750,-1.553406,-1.330009,0.276400,1.418005,0.431737,-0.791542,1.228754,-1.673899,1,2,2
861,-1.534305,1.469779,-0.716407,-1.381093,-0.581554,-0.256538,2.030889,-0.893509,1.539899,1,1,1


In [21]:
# Exportamos el dataset limpio
X_train.to_csv("data/Data_Limpia_y_Separada/X_train.csv", index=False)
X_test.to_csv("data/Data_Limpia_y_Separada/X_test.csv", index=False)
y_train.to_csv("data/Data_Limpia_y_Separada/y_train.csv", index=False)
y_test.to_csv("data/Data_Limpia_y_Separada/y_test.csv", index=False)
X_val.to_csv("data/Data_Limpia_y_Separada/X_val.csv", index=False)
y_val.to_csv("data/Data_Limpia_y_Separada/y_val.csv", index=False)